# Daily Menu Insights (cuaderno de prueba)

Este cuaderno reutiliza los módulos del proyecto. No envía correos ni llama a un LLM si no se configura expresamente un proveedor.

## 1. Configuración

In [ ]:
from datetime import date
from pathlib import Path
from database import MenuDatabase
from models import DishRecord, ExtractionSummary, MenuRow, Recommendations
from scraper import ExtractionResult, load_env_file, scrape_date, persist_extraction
from nutrition import enrich_missing_nutrition, recommend_daily, _prompt_for_dish
from mailer import DailyReport, build_report, save_preview, send_report

load_env_file()
run_date: str = date.today().isoformat()  # Puede sustituirse por '2026-10-20'
db: MenuDatabase = MenuDatabase('data/menus.db')
print(f'Base SQLite: {db.path.resolve()} | fecha: {run_date}')

## 2. Scraping

La página pública se consulta una vez y se parsea su JSON hidratado.

In [ ]:
result: ExtractionResult = scrape_date(date.fromisoformat(run_date))
[(item['name'], item['status'], len(item['dishes'])) for item in result.restaurants.values()]

In [ ]:
green_roots: list[DishRecord] = result.restaurants[5]['dishes']
[{key: dish[key] for key in ('name', 'ingredients', 'groups', 'categories', 'published_nutrition')} for dish in green_roots[:5]]

## 3. SQLite

La segunda ejecución de esta celda no duplica platos ni disponibilidades.

In [ ]:
persist_extraction(db, result)
rows: list[MenuRow] = db.menu_for_date(run_date)
summary: list[ExtractionSummary] = db.extraction_summary(run_date)
len(rows), summary

## 4. Análisis LLM

El prompt recibe datos oficiales y solo completa huecos. Dejarlo en `none` para trabajar sin LLM.  Usar `gemini` solo después de configurar `.env`.

In [ ]:
sample: MenuRow = rows[0]
print(_prompt_for_dish(sample))
provider: str = 'none'
enrich_missing_nutrition(db, rows, provider_name=provider)
rows = db.menu_for_date(run_date)
{'oficial': sample['published_nutrition'], 'estimacion_cacheada': rows[0]['estimate']}

## 5. Recomendaciones

In [ ]:
recommendations: Recommendations = recommend_daily(rows, calorie_target=700, protein_target=35)
{key: recommendations[key] for key in ('balanced', 'high_protein', 'light', 'recovery', 'custom')}

## 6. Generación del email

Comprueba que las cuatro secciones y todos los platos aparezcan antes de enviar.

In [ ]:
report: DailyReport = build_report(run_date, result.extracted_at, rows, summary, recommendations)
preview: Path = save_preview(report, f'output/menu_{run_date}.html')
assert all(dish['name'] in report.html for dish in rows)
assert len(summary) == len(result.restaurants)
preview.resolve()

In [ ]:
from IPython.display import IFrame, display
display(IFrame(src=str(preview), width='100%', height=700))

## 7. Prueba de envío

Por seguridad, no se ejecuta salvo que cambies explícitamente la variable siguiente a `True` y hayas configurado SMTP en `.env`.

In [ ]:
SEND_TEST_EMAIL: bool = False
if SEND_TEST_EMAIL:
    sent: bool = send_report(db, report, force=True, preview_path=preview)
    print('SMTP aceptó el mensaje:', sent)
else:
    print('No enviado. Cambia SEND_TEST_EMAIL a True únicamente para una prueba explícita.')